# Notebook 12 -- Transpilation and Hardware Constraints

**Prerequisites:** Notebooks 01-06. Familiarity with multi-qubit gates.

Real quantum hardware is not a universal instruction set machine. Each
processor supports only a small set of **basis gates** (e.g., CX + RZ + SX
on IBM, or CZ + RZ + RX on Google) and has limited **qubit connectivity**
(most qubits can only interact with their nearest neighbors).

**Transpilation** is the process of converting an abstract quantum circuit
into one that respects these hardware constraints:

1. **Gate decomposition** -- replace high-level gates (Toffoli, SWAP, etc.)
   with sequences of basis gates the hardware can execute.
2. **Qubit routing** -- insert SWAP gates so that two-qubit gates only act
   on physically connected qubits.
3. **Optimization** -- cancel redundant gates, merge rotations, and
   parallelize operations to reduce depth and error.

By the end of this notebook you will be able to:

1. **Describe** hardware constraints (basis gates, connectivity) for different quantum processors.
2. **Implement** transpilation to specific hardware targets at different optimization levels.
3. **Explain** Euler and KAK gate decomposition.
4. **Verify** that transpilation preserves circuit functionality.

In this notebook we will:

- Explore hardware targets and their basis gates.
- Transpile circuits with `pipeline.Run` at different optimization levels.
- Decompose single-qubit gates (Euler) and two-qubit gates (KAK).
- Route qubits with the SABRE algorithm.
- Apply individual transpilation passes.
- Verify that transpilation preserves circuit functionality.

In [1]:
import (
	"context"
	"fmt"

	"github.com/janpfeifer/gonb/gonbui"
	"github.com/splch/goqu/circuit/builder"
	"github.com/splch/goqu/circuit/draw"
	"github.com/splch/goqu/circuit/gate"
	"github.com/splch/goqu/circuit/ir"
	"github.com/splch/goqu/transpile/decompose"
	"github.com/splch/goqu/transpile/pass"
	"github.com/splch/goqu/transpile/pipeline"
	"github.com/splch/goqu/transpile/routing"
	"github.com/splch/goqu/transpile/target"
	"github.com/splch/goqu/transpile/verify"
	"github.com/splch/goqu/viz"
)

## Hardware Targets

Each hardware target in Goqu is described by a `target.Target` struct with:

- **Name** -- the hardware identifier.
- **NumQubits** -- number of physical qubits.
- **BasisGates** -- the native gate set the processor can execute.
- **Connectivity** -- which qubit pairs are physically connected
  (`nil` means all-to-all, as in trapped-ion machines).

Different hardware families have fundamentally different native gates:

| Family | 2Q Gate | 1Q Gates | Connectivity |
|--------|---------|----------|-------------|
| IBM (superconducting) | CX | RZ, SX, X | Heavy-hex lattice |
| Google (superconducting) | CZ | RZ, RX | 2D grid |
| IonQ (trapped ion) | MS | GPI, GPI2 | All-to-all |
| Quantinuum (trapped ion) | RZZ | RZ, RY | All-to-all |
| Rigetti (superconducting) | CZ | RX, RZ | Square-octagon lattice |

Let's inspect the available targets.

In [2]:
%%
targets := []target.Target{
	target.IBMBrisbane,
	target.IBMBrisbane,
	target.GoogleSycamore,
	target.IonQAria,
	target.QuantinuumH1,
	target.RigettiAnkaa,
}

fmt.Println("Available Hardware Targets")
fmt.Println("=========================")
for _, t := range targets {
	connectivity := "all-to-all"
	if t.Connectivity != nil {
		connectivity = fmt.Sprintf("%d edges", len(t.Connectivity))
	}
	fmt.Printf("%-20s  basis=%-28v  qubits=%d  connectivity=%s\n",
		t.Name, t.BasisGates, t.NumQubits, connectivity)
}

fmt.Println("\nNote: trapped-ion machines (IonQ, Quantinuum) have all-to-all connectivity,")
fmt.Println("meaning any two qubits can interact directly without SWAP routing.")

Available Hardware Targets
ibm.brisbane          basis=[CX                           RZ                           SX                           X                            I                           ]  qubits=127  connectivity=all-to-all
ibm.brisbane          basis=[CX                           RZ                           SX                           X                            I                           ]  qubits=127  connectivity=all-to-all
Google Sycamore       basis=[CZ                           RZ                           RX                          ]  qubits=53  connectivity=all-to-all
IonQ Aria             basis=[GPI                          GPI2                         MS                          ]  qubits=25  connectivity=all-to-all
Quantinuum H1         basis=[RZZ                          RZ                           RY                          ]  qubits=20  connectivity=all-to-all
Rigetti Ankaa-3       basis=[CZ                           RX                           RZ 

## Transpilation with `pipeline.Run`

The `pipeline.Run` function is the main entry point for transpilation. It
takes an abstract circuit and a hardware target, then applies a sequence of
passes (decomposition, routing, optimization) to produce a hardware-compatible
circuit.

```go
out, err := pipeline.Run(ctx, circuit, target, level)
```

Let's transpile a Toffoli (CCX) gate -- a 3-qubit gate that is not in any
hardware's native basis set -- to the IBM Eagle target.

In [3]:
%%
c, _ := builder.New("toffoli", 3).CCX(0, 1, 2).Build()

fmt.Println("Before transpilation:")
gonbui.DisplayHTML(draw.SVG(c))
fmt.Printf("Stats: depth=%d, gates=%d, 2Q=%d\n\n",
	c.Stats().Depth, c.Stats().GateCount, c.Stats().TwoQubitGates)

ctx := context.Background()
out, err := pipeline.Run(ctx, c, target.IBMBrisbane, pipeline.LevelFull)
if err != nil {
	fmt.Println("Error:", err)
} else {
	fmt.Println("After transpilation (IBM Brisbane, LevelFull):")
	gonbui.DisplayHTML(draw.SVG(out))
	fmt.Printf("Stats: depth=%d, gates=%d, 2Q=%d\n",
		out.Stats().Depth, out.Stats().GateCount, out.Stats().TwoQubitGates)
	fmt.Println("\nThe single CCX gate has been decomposed into CX + single-qubit rotations.")
}

Before transpilation:
Stats: depth=1, gates=1, 2Q=1



q0 
 
 q1 
 
 q2

After transpilation (IBM Brisbane, LevelFull):
Stats: depth=13, gates=18, 2Q=6

The single CCX gate has been decomposed into CX + single-qubit rotations.


q0 
 
 q1 
 
 q2 
 
 RZ(pi/2) 
 SX 
 RZ(pi/2) 
 
 
 
 RZ(-pi/4) 
 
 
 
 RZ(pi/4) 
 
 
 
 RZ(-pi/4) 
 RZ(pi/4) 
 
 
 
 RZ(3*pi/4) 
 
 
 
 SX 
 RZ(pi/4) 
 RZ(-pi/4) 
 RZ(pi/2)

## Optimization Levels

The transpiler supports multiple optimization levels that trade compilation
time for circuit quality:

| Level | Name | Passes |
|-------|------|--------|
| 0 | `LevelNone` | Decompose + fix direction + validate only |
| 1 | `LevelBasic` | + routing + cancel adjacent + merge rotations |
| 2 | `LevelFull` | + commutation + parallelization |

Higher levels produce shorter, lower-error circuits at the cost of longer
compilation time. Let's compare all three on the same Toffoli circuit.

In [4]:
%%
c, _ := builder.New("toffoli", 3).CCX(0, 1, 2).Build()
ctx := context.Background()

fmt.Println("Optimization Level Comparison (Toffoli on IBM Brisbane)")
fmt.Println("=====================================================")
fmt.Printf("%-12s  %6s  %6s  %4s\n", "Level", "Depth", "Gates", "2Q")
fmt.Println("--------------------------------------------")

for _, level := range []pipeline.Level{pipeline.LevelNone, pipeline.LevelBasic, pipeline.LevelFull} {
	out, err := pipeline.Run(ctx, c, target.IBMBrisbane, level)
	if err != nil {
		fmt.Printf("Level %d: error: %v\n", level, err)
		continue
	}
	stats := out.Stats()
	name := []string{"LevelNone", "LevelBasic", "LevelFull"}[level]
	fmt.Printf("%-12s  %6d  %6d  %4d\n", name, stats.Depth, stats.GateCount, stats.TwoQubitGates)
}

fmt.Println("\nHigher optimization levels reduce depth and gate count by")
fmt.Println("canceling redundant gates and merging rotations.")

Optimization Level Comparison (Toffoli on IBM Brisbane)
Level          Depth   Gates    2Q
--------------------------------------------
LevelNone         14      19     6
LevelBasic        13      18     6
LevelFull         13      18     6

Higher optimization levels reduce depth and gate count by
canceling redundant gates and merging rotations.


## Gate Decomposition: Euler Decomposition

Any single-qubit gate (a 2x2 unitary) can be decomposed into a sequence of
three rotations using **Euler angles**. The most common convention is ZYZ:

$$U = e^{i\phi} \cdot R_Z(\alpha) \cdot R_Y(\beta) \cdot R_Z(\gamma)$$

Different hardware targets prefer different Euler bases:

- **IBM**: ZSX basis (RZ, SX, X) -- converts ZYZ angles to {RZ, SX} sequences.
- **Google / Rigetti**: ZXZ basis (RZ, RX).
- **Quantinuum**: ZYZ basis (RZ, RY) -- native.

The `decompose.EulerDecompose` function decomposes any 1-qubit gate into
RZ and RY rotations (ZYZ convention). Let's decompose a U3 gate.

In [5]:
%%
// Euler decomposition of an arbitrary single-qubit gate U3(1.0, 2.0, 3.0)
fmt.Println("Euler Decomposition (ZYZ Convention)")
fmt.Println("=====================================")

g := gate.U3(1.0, 2.0, 3.0)
fmt.Printf("Original gate: %s\n\n", g.Name())

ops := decompose.EulerDecompose(g, 0)
fmt.Println("Decomposed into:")
for _, op := range ops {
	fmt.Printf("  %s on qubit %v\n", op.Gate.Name(), op.Qubits)
}

fmt.Println("\nAny 1-qubit unitary can be written as RZ(alpha) . RY(beta) . RZ(gamma).")
fmt.Println("Identity rotations (angle ~ 0 mod 2pi) are automatically skipped.")

Euler Decomposition (ZYZ Convention)
Original gate: U3(1.0000,2.0000,3.0000)

Decomposed into:
  RZ(3.0000) on qubit [0]
  RY(1.0000) on qubit [0]
  RZ(2.0000) on qubit [0]

Any 1-qubit unitary can be written as RZ(alpha) . RY(beta) . RZ(gamma).
Identity rotations (angle ~ 0 mod 2pi) are automatically skipped.


## Gate Decomposition: KAK Decomposition

For two-qubit gates, the **KAK (Cartan) decomposition** factorizes any 4x4
unitary into at most 3 CNOT gates plus single-qubit rotations:

$$U = (K_{1L} \otimes K_{1R}) \cdot \exp\big(i(x\,XX + y\,YY + z\,ZZ)\big) \cdot (K_{2L} \otimes K_{2R})$$

The number of nonzero Weyl parameters (x, y, z) determines the minimum
CNOT count:

| Nonzero params | CNOTs needed | Example gates |
|:-:|:-:|:-|
| 0 | 0 | A tensor B (no entanglement) |
| 1 (= pi/4) | 1 | CNOT, CZ, CY |
| 1 (other) | 2 | Partial entanglers |
| 2-3 | 3 | SWAP, general 2Q gates |

Let's decompose the CNOT gate matrix using KAK.

In [6]:
%%
// KAK decomposition of the CNOT matrix
fmt.Println("KAK Decomposition of CNOT")
fmt.Println("=========================")

ops := decompose.KAK(gate.CNOT.Matrix(), 0, 1)
fmt.Println("Decomposed into:")
for _, op := range ops {
	fmt.Printf("  %s on qubit %v\n", op.Gate.Name(), op.Qubits)
}
fmt.Printf("Total operations: %d\n", len(ops))

fmt.Println("\nCNOT is already a minimal 1-CNOT gate (it IS a CNOT), so KAK")
fmt.Println("recognizes it and returns a single CNOT operation.")

// Now decompose SWAP -- should require 3 CNOTs
fmt.Println("\nKAK Decomposition of SWAP")
fmt.Println("=========================")

swapOps := decompose.KAK(gate.SWAP.Matrix(), 0, 1)
fmt.Println("Decomposed into:")
cxCount := 0
for _, op := range swapOps {
	fmt.Printf("  %s on qubit %v\n", op.Gate.Name(), op.Qubits)
	if op.Gate.Name() == "CNOT" {
		cxCount++
	}
}
fmt.Printf("CNOT count: %d (the theoretical minimum for SWAP)\n", cxCount)

KAK Decomposition of CNOT
Decomposed into:
  CNOT on qubit [0 1]
Total operations: 1

CNOT is already a minimal 1-CNOT gate (it IS a CNOT), so KAK
recognizes it and returns a single CNOT operation.

KAK Decomposition of SWAP
Decomposed into:
  CNOT on qubit [0 1]
  CNOT on qubit [1 0]
  CNOT on qubit [0 1]
CNOT count: 3 (the theoretical minimum for SWAP)


## Qubit Routing with SABRE

On hardware with limited connectivity, a CNOT between non-adjacent qubits
cannot be executed directly. The transpiler must insert **SWAP gates** to
move qubit states to neighboring positions. Each SWAP costs 3 CNOTs, so
minimizing SWAPs is critical.

Goqu uses the **SABRE** algorithm (Li et al., arXiv:1809.02573) for qubit
routing. SABRE is a heuristic that:

1. Runs multiple random initial layouts in parallel.
2. Uses bidirectional iteration (forward + backward) to refine layouts.
3. Employs lookahead to choose SWAPs that help future gates, not just the
   current one.

For all-to-all targets (IonQ, Quantinuum), routing is a no-op -- every
qubit pair is directly connected.

In [7]:
%%
// Route a circuit with a non-adjacent CNOT onto a small linear topology.
// We define a 5-qubit linear chain: 0-1-2-3-4
smallLinear := target.Target{
	Name:       "Linear-5",
	NumQubits:  5,
	BasisGates: []string{"CX", "RZ", "SX", "X"},
	Connectivity: []target.QubitPair{
		{0, 1}, {1, 2}, {2, 3}, {3, 4},
	},
}

c, _ := builder.New("route-demo", 4).CNOT(0, 3).CNOT(1, 2).Build()

fmt.Println("Original circuit:")
gonbui.DisplayHTML(draw.SVG(c))
fmt.Printf("Stats: depth=%d, gates=%d, 2Q=%d\n\n",
	c.Stats().Depth, c.Stats().GateCount, c.Stats().TwoQubitGates)

routed, err := routing.Route(c, smallLinear)
if err != nil {
	fmt.Println("Error:", err)
} else {
	fmt.Println("Routed circuit (Linear-5):")
	gonbui.DisplayHTML(draw.SVG(routed))
	fmt.Printf("Stats: depth=%d, gates=%d, 2Q=%d\n",
		routed.Stats().Depth, routed.Stats().GateCount, routed.Stats().TwoQubitGates)
	fmt.Println("\nSWAP gates were inserted to satisfy connectivity constraints.")
	fmt.Println("The CNOT(0,3) required routing because qubits 0 and 3 are not adjacent.")
}

Original circuit:
Stats: depth=1, gates=2, 2Q=2



q0 
 
 q1 
 
 q2 
 
 q3

Routed circuit (Linear-5):
Stats: depth=1, gates=2, 2Q=2

SWAP gates were inserted to satisfy connectivity constraints.
The CNOT(0,3) required routing because qubits 0 and 3 are not adjacent.


q0 
 
 q1 
 
 q2 
 
 q3

## Transpiling to Different Targets

The same abstract circuit compiles to very different hardware-specific
circuits depending on the target. Let's transpile a Toffoli gate to three
different hardware families and compare the results.

Key differences to watch for:
- **IBM** uses CX as its 2Q gate, with {RZ, SX, X} for 1Q.
- **Google** uses CZ as its 2Q gate, with {RZ, RX} for 1Q.
- **IonQ** uses MS (Molmer-Sorensen) as its 2Q gate, with {GPI, GPI2} for 1Q.
  Because IonQ has all-to-all connectivity, no SWAP routing is needed.

In [8]:
%%
c, _ := builder.New("toffoli", 3).CCX(0, 1, 2).Build()
ctx := context.Background()

fmt.Println("Cross-Target Transpilation: Toffoli Gate")
fmt.Println("=========================================")
fmt.Printf("\nOriginal: depth=%d, gates=%d, 2Q=%d\n\n",
	c.Stats().Depth, c.Stats().GateCount, c.Stats().TwoQubitGates)

crossTargets := []target.Target{
	target.IBMBrisbane,
	target.GoogleSycamore,
	target.IonQAria,
}

fmt.Printf("%-20s  %6s  %6s  %4s  %s\n", "Target", "Depth", "Gates", "2Q", "Basis")
fmt.Println("---------------------------------------------------------------")

for _, t := range crossTargets {
	out, err := pipeline.Run(ctx, c, t, pipeline.LevelFull)
	if err != nil {
		fmt.Printf("%-20s  error: %v\n", t.Name, err)
		continue
	}
	stats := out.Stats()
	fmt.Printf("%-20s  %6d  %6d  %4d  %v\n",
		t.Name, stats.Depth, stats.GateCount, stats.TwoQubitGates, t.BasisGates)
}

fmt.Println("\nThe same logical circuit produces very different physical circuits")
fmt.Println("depending on the hardware's native gate set and connectivity.")

Cross-Target Transpilation: Toffoli Gate

Original: depth=1, gates=1, 2Q=1

Target                 Depth   Gates    2Q  Basis
---------------------------------------------------------------
ibm.brisbane              13      18     6  [CX RZ SX X I]
Google Sycamore       error: decompose: max depth exceeded for gate "CNOT" on qubits [1 2]
IonQ Aria                 27      46     6  [GPI GPI2 MS]

The same logical circuit produces very different physical circuits
depending on the hardware's native gate set and connectivity.


## Individual Transpilation Passes

The pipeline is composed of individual **passes**, each performing one
specific transformation. You can apply them individually for fine-grained
control:

| Pass | Function | Purpose |
|------|----------|---------|
| Cancel Adjacent | `pass.CancelAdjacent` | Remove adjacent inverse pairs (H.H = I) |
| Merge Rotations | `pass.MergeRotations` | Combine RZ(a).RZ(b) into RZ(a+b) |
| Commute | `pass.CommuteThroughCNOT` | Push gates through CNOTs to enable cancellation |
| Parallelize | `pass.ParallelizeOps` | Reorder independent ops to minimize depth |
| Remove Barriers | `pass.RemoveBarriers` | Strip barrier pseudo-gates |

Each pass has the signature `func(*ir.Circuit, target.Target) (*ir.Circuit, error)`.

Let's see `CancelAdjacent` in action: two consecutive H gates should cancel
to identity because $H \cdot H = I$.

In [9]:
%%
// Cancel adjacent inverse gates: H . H = I
c, _ := builder.New("cancel", 2).H(0).H(0).CNOT(0, 1).Build()

fmt.Println("CancelAdjacent Pass")
fmt.Println("===================")
fmt.Println("\nBefore:")
gonbui.DisplayHTML(draw.SVG(c))
fmt.Printf("Gates: %d\n\n", c.Stats().GateCount)

canceled, err := pass.CancelAdjacent(c, target.Simulator)
if err != nil {
	fmt.Println("Error:", err)
} else {
	fmt.Println("After:")
	gonbui.DisplayHTML(draw.SVG(canceled))
	fmt.Printf("Gates: %d\n", canceled.Stats().GateCount)
	fmt.Println("\nThe two adjacent H gates were recognized as inverses and removed.")
}

CancelAdjacent Pass

Before:
Gates: 3

After:
Gates: 1

The two adjacent H gates were recognized as inverses and removed.


q0 
 
 q1 
 
 H 
 H

q0 
 
 q1

In [10]:
%%
// MergeRotations: combine consecutive same-axis rotations
c, _ := builder.New("merge", 1).RZ(0.5, 0).RZ(0.3, 0).RZ(0.2, 0).Build()

fmt.Println("MergeRotations Pass")
fmt.Println("===================")
fmt.Println("\nBefore:")
gonbui.DisplayHTML(draw.SVG(c))
fmt.Printf("Gates: %d\n\n", c.Stats().GateCount)

merged, err := pass.MergeRotations(c, target.Simulator)
if err != nil {
	fmt.Println("Error:", err)
} else {
	fmt.Println("After:")
	gonbui.DisplayHTML(draw.SVG(merged))
	fmt.Printf("Gates: %d\n", merged.Stats().GateCount)
	fmt.Println("\nThree consecutive RZ gates merged into one: RZ(0.5+0.3+0.2) = RZ(1.0)")
}

MergeRotations Pass

Before:


q0 
 
 RZ(0.5) 
 RZ(0.3) 
 RZ(0.2)

q0 
 
 RZ(1)

Gates: 3

After:
Gates: 1

Three consecutive RZ gates merged into one: RZ(0.5+0.3+0.2) = RZ(1.0)


## Predict, Then Verify

**Question:** How many CX (CNOT) gates does a Toffoli gate decompose into
on IBM Eagle at each optimization level?

**Prediction:** The standard Toffoli decomposition uses 6 CNOTs. With
optimization:
- `LevelNone`: should produce the raw decomposition (6 CX gates).
- `LevelBasic`: cancellation and merging may reduce the count.
- `LevelFull`: commutation and parallelization may help further.

Let's verify.

In [11]:
%%
c, _ := builder.New("toffoli", 3).CCX(0, 1, 2).Build()
ctx := context.Background()

fmt.Println("How many CX gates does a Toffoli require on IBM Brisbane?")
fmt.Println("======================================================")

for _, level := range []pipeline.Level{pipeline.LevelNone, pipeline.LevelBasic, pipeline.LevelFull} {
	out, err := pipeline.Run(ctx, c, target.IBMBrisbane, level)
	if err != nil {
		fmt.Printf("Level %d: error: %v\n", level, err)
		continue
	}
	stats := out.Stats()
	name := []string{"LevelNone", "LevelBasic", "LevelFull"}[level]
	fmt.Printf("%s: %d CX gates, depth=%d, total gates=%d\n",
		name, stats.TwoQubitGates, stats.Depth, stats.GateCount)
}

fmt.Println("\nOptimization reduces the CX count by canceling redundant gates")
fmt.Println("that appear after the initial decomposition.")

How many CX gates does a Toffoli require on IBM Brisbane?
LevelNone: 6 CX gates, depth=14, total gates=19
LevelBasic: 6 CX gates, depth=13, total gates=18
LevelFull: 6 CX gates, depth=13, total gates=18

Optimization reduces the CX count by canceling redundant gates
that appear after the initial decomposition.


## Self-Check Questions

Before attempting the exercises, make sure you can answer these:

1. Why do trapped-ion machines need no SWAP routing while superconducting machines do?
2. How many CNOT gates does each SWAP gate cost, and why does this matter for circuit fidelity?
3. Why can the same abstract circuit produce vastly different gate counts on different hardware targets?

---

## Exercises

### Exercise 1 -- Cross-Target Comparison

Build a 3-qubit QFT circuit (H, controlled-phase, SWAP) and transpile it
to IBM Eagle, Google Sycamore, and Quantinuum H1. Compare the gate counts
and depths. Which target produces the most efficient circuit, and why?

In [12]:
%%
// Exercise 1: Transpile a small QFT to multiple targets
// Build a 3-qubit QFT circuit: H, controlled-phase, SWAP
//
// TODO: Build the 3-qubit QFT circuit
// import "math"
// c, _ := builder.New("qft3", 3).
//     H(0).
//     Ctrl(gate.Phase(math.Pi/2), []int{1}, 0).
//     Ctrl(gate.Phase(math.Pi/4), []int{2}, 0).
//     H(1).
//     Ctrl(gate.Phase(math.Pi/2), []int{2}, 1).
//     H(2).
//     SWAP(0, 2).
//     Build()
//
// TODO: Transpile to IBM Eagle, Google Sycamore, and Quantinuum H1
// For each target:
//   out, err := pipeline.Run(ctx, c, target.???, pipeline.LevelFull)
//   Print depth, gate count, and 2Q gate count
//
// TODO: Compare results and explain why all-to-all targets
//       (Quantinuum) avoid SWAP routing overhead
_ = context.Background()
fmt.Println("Uncomment the code above and build the QFT circuit!")

Uncomment the code above and build the QFT circuit!


### Exercise 2 -- Verify Transpilation Preserves Functionality

Transpilation should never change **what** a circuit computes -- only
**how** it computes it. Use `verify.Equivalent` to confirm that the
transpiled circuit produces the same unitary as the original.

In [13]:
%%
// Exercise 2: Verify transpilation preserves the unitary
// Transpile a Toffoli gate to multiple targets and confirm equivalence
//
// TODO: Build a Toffoli circuit
// c, _ := builder.New("toffoli", 3).CCX(0, 1, 2).Build()
//
// TODO: For each target (IBM Brisbane, Google Sycamore, IonQ Aria):
//   1. Transpile with pipeline.Run at LevelFull
//   2. Verify equivalence using verify.Equivalent(c, out, 1e-6)
//   3. Print whether the circuits are equivalent and the gate stats
//
// for _, t := range []target.Target{target.IBMBrisbane, target.GoogleSycamore, target.IonQAria} {
//     out, err := pipeline.Run(ctx, c, t, pipeline.LevelFull)
//     eq, err := verify.Equivalent(c, out, ???)
//     fmt.Printf("%s: equivalent=%v  (gates=%d, 2Q=%d)\n",
//         t.Name, eq, out.Stats().GateCount, out.Stats().TwoQubitGates)
// }
_ = context.Background()
fmt.Println("Uncomment the code above and verify transpilation!")

Uncomment the code above and verify transpilation!


## Key Takeaways

1. **Real hardware has constraints**: each processor supports only a few
   basis gates and has limited qubit connectivity. Transpilation bridges
   the gap between abstract circuits and physical execution.

2. **Gate decomposition** breaks high-level gates into basis gates. Euler
   decomposition handles single-qubit gates (at most 3 rotations); KAK
   decomposition handles two-qubit gates (at most 3 CNOTs).

3. **Qubit routing** (SABRE) inserts SWAP gates to satisfy connectivity.
   Each SWAP costs 3 CNOTs, so all-to-all targets (trapped ions) have a
   significant advantage for circuits with long-range interactions.

4. **Optimization levels** trade compilation time for circuit quality.
   Higher levels (LevelFull) apply more passes: cancellation, merging,
   commutation, and parallelization.

5. **Individual passes** give fine-grained control. `CancelAdjacent`
   removes inverse pairs, `MergeRotations` combines same-axis rotations,
   `CommuteThroughCNOT` moves gates through entangling gates, and
   `ParallelizeOps` reorders for minimum depth.

6. **Transpilation preserves functionality**: the `verify` package confirms
   that transpiled circuits compute the same unitary as the original,
   up to global phase.

7. **Target choice matters**: the same abstract circuit can produce vastly
   different physical circuits depending on the hardware. Understanding
   your target's basis gates and connectivity is essential for writing
   efficient quantum programs.

---

**Next up:** Notebook 13 — Variational Algorithms, where we'll build parameterized circuits and use classical-quantum hybrid loops for VQE and QAOA.